In [2]:

!pip install -q --upgrade transformers datasets accelerate scikit-learn evaluate


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 512.3/512.3 kB 14.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 135.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 9.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.7/47.7 MB 22.6 MB/s eta 0:00:00


In [3]:
import torch
import gc
import numpy as np
import pandas as pd
from datasets import load_dataset
import evaluate
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer
)

In [5]:
model = AutoModelForSequenceClassification.from_pretrained("answerdotai/ModernBERT-base")


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/599M [00:00<?, ?B/s]

Some weights of ModernBertForSequenceClassification were not initialized from the model checkpoint at answerdotai/ModernBERT-base and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [8]:
from torchsummary import summary
model

ModernBertForSequenceClassification(
  (model): ModernBertModel(
    (embeddings): ModernBertEmbeddings(
      (tok_embeddings): Embedding(50368, 768, padding_idx=50283)
      (norm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
      (drop): Dropout(p=0.0, inplace=False)
    )
    (layers): ModuleList(
      (0): ModernBertEncoderLayer(
        (attn_norm): Identity()
        (attn): ModernBertAttention(
          (Wqkv): Linear(in_features=768, out_features=2304, bias=False)
          (rotary_emb): ModernBertRotaryEmbedding()
          (Wo): Linear(in_features=768, out_features=768, bias=False)
          (out_drop): Identity()
        )
        (mlp_norm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (mlp): ModernBertMLP(
          (Wi): Linear(in_features=768, out_features=2304, bias=False)
          (act): GELUActivation()
          (drop): Dropout(p=0.0, inplace=False)
          (Wo): Linear(in_features=1152, out_features=768, bias=False)
        )
      

In [12]:
  model = AutoModelForSequenceClassification.from_pretrained("answerdotai/ModernBERT-base", num_labels=11)
  model

Some weights of ModernBertForSequenceClassification were not initialized from the model checkpoint at answerdotai/ModernBERT-base and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


ModernBertForSequenceClassification(
  (model): ModernBertModel(
    (embeddings): ModernBertEmbeddings(
      (tok_embeddings): Embedding(50368, 768, padding_idx=50283)
      (norm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
      (drop): Dropout(p=0.0, inplace=False)
    )
    (layers): ModuleList(
      (0): ModernBertEncoderLayer(
        (attn_norm): Identity()
        (attn): ModernBertAttention(
          (Wqkv): Linear(in_features=768, out_features=2304, bias=False)
          (rotary_emb): ModernBertRotaryEmbedding()
          (Wo): Linear(in_features=768, out_features=768, bias=False)
          (out_drop): Identity()
        )
        (mlp_norm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (mlp): ModernBertMLP(
          (Wi): Linear(in_features=768, out_features=2304, bias=False)
          (act): GELUActivation()
          (drop): Dropout(p=0.0, inplace=False)
          (Wo): Linear(in_features=1152, out_features=768, bias=False)
        )
      

In [9]:
for name, module in model.named_modules():
    print(name)


model
model.embeddings
model.embeddings.tok_embeddings
model.embeddings.norm
model.embeddings.drop
model.layers
model.layers.0
model.layers.0.attn_norm
model.layers.0.attn
model.layers.0.attn.Wqkv
model.layers.0.attn.rotary_emb
model.layers.0.attn.Wo
model.layers.0.attn.out_drop
model.layers.0.mlp_norm
model.layers.0.mlp
model.layers.0.mlp.Wi
model.layers.0.mlp.act
model.layers.0.mlp.drop
model.layers.0.mlp.Wo
model.layers.1
model.layers.1.attn_norm
model.layers.1.attn
model.layers.1.attn.Wqkv
model.layers.1.attn.rotary_emb
model.layers.1.attn.Wo
model.layers.1.attn.out_drop
model.layers.1.mlp_norm
model.layers.1.mlp
model.layers.1.mlp.Wi
model.layers.1.mlp.act
model.layers.1.mlp.drop
model.layers.1.mlp.Wo
model.layers.2
model.layers.2.attn_norm
model.layers.2.attn
model.layers.2.attn.Wqkv
model.layers.2.attn.rotary_emb
model.layers.2.attn.Wo
model.layers.2.attn.out_drop
model.layers.2.mlp_norm
model.layers.2.mlp
model.layers.2.mlp.Wi
model.layers.2.mlp.act
model.layers.2.mlp.drop
mod

In [3]:

BATCH_SIZE = 16

MODEL_NAME = "answerdotai/ModernBERT-base"
FREEZE_RATIO = 0.60
EPOCHS = 3
LEARNING_RATE = 2e-5

# All TweetEval sub-tasks
TASKS = [
    'emotion',
    'sentiment',
    'hate',
    'irony',
    'offensive',
    'emoji',
    'stance_abortion',
    'stance_atheism',
    'stance_climate',
    'stance_feminism',
    'stance_hillary'
]



In [8]:

# Trainer and other functions


def compute_metrics(eval_pred):
    metric_f1 = evaluate.load("f1")
    metric_acc = evaluate.load("accuracy")
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)

    #
    f1 = metric_f1.compute(predictions=predictions, references=labels, average="macro")
    acc = metric_acc.compute(predictions=predictions, references=labels)
    return {"macro_f1": f1["f1"], "accuracy": acc["accuracy"]}

def run_task(task_name, tokenizer):
    print(f"\n{'='*40}")
    print(f">>> PROCESSING TASK: {task_name}")
    print(f"{'='*40}")

    # Load data
    try:
        dataset = load_dataset("tweet_eval", task_name)
    except Exception as e:
        print(f"Skipping {task_name} due to error: {e}")
        return None

    # Identify Labels
    label_list = dataset['train'].features['label'].names
    num_labels = len(label_list)

    # Tokenization
    def tokenize_function(examples):
        return tokenizer(examples["text"], truncation=True, padding="max_length", max_length=128)

    tokenized_datasets = dataset.map(tokenize_function, batched=True)

    # Load Model (Fresh init)
    model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=num_labels)


    # FREEZING 60% of 22 Layers ( = 13 Layers)

    total_layers = model.config.num_hidden_layers
    layers_to_freeze = int(total_layers * FREEZE_RATIO)

    print(f"Freezing embeddings + first {layers_to_freeze} encoder layers...")

    # Freeze Embeddings
    for param in model.model.embeddings.parameters():
        param.requires_grad = False

    # Freeze Encoder Layers (0 to 12)
    for i in range(layers_to_freeze):
        for param in model.model.layers[i].parameters():
            param.requires_grad = False

    # Trainer Setup
    training_args = TrainingArguments(
        output_dir=f"./results/{task_name}",
        eval_strategy="epoch",
        save_strategy="epoch",
        learning_rate=LEARNING_RATE,
        per_device_train_batch_size=BATCH_SIZE,
        per_device_eval_batch_size=BATCH_SIZE,
        num_train_epochs=EPOCHS,
        weight_decay=0.01,
        load_best_model_at_end=True,
        metric_for_best_model="macro_f1",
        fp16=torch.cuda.is_available(),
        logging_steps=50,
        report_to="none",
        save_total_limit=1 # Save disk space
    )

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=tokenized_datasets["train"],
        eval_dataset=tokenized_datasets["validation"],
        compute_metrics=compute_metrics,
    )

    # Train
    trainer.train()

    # Evaluate on Test
    print(f"Evaluating {task_name} on Test Set...")
    test_results = trainer.evaluate(tokenized_datasets["test"])


    # MEMORY CLEANUP

    del model
    del trainer
    torch.cuda.empty_cache()
    gc.collect()

    return test_results



In [9]:

# MAIN EXECUTION LOOP

def main():
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
    all_results = []

    for task in TASKS:
        result = run_task(task, tokenizer)
        if result:
            all_results.append({
                "Task": task,
                "Macro F1": round(result['eval_macro_f1'], 4),
                "Accuracy": round(result['eval_accuracy'], 4),
                "Loss": round(result['eval_loss'], 4)
            })

    # FINAL REPORT
    print("\n\n" + "#"*40)
    print(f"FINAL RESULTS (Batch Size: {BATCH_SIZE})")
    print("#"*40)

    df = pd.DataFrame(all_results)
    print(df.to_string(index=False))

    # Save to CSV for download
    # csv_filename = f"modernbert_results_bs{BATCH_SIZE}.csv"
    # df.to_csv(csv_filename, index=False)
    # print(f"\nSaved results to {csv_filename}")

    # if not df.empty:
    #     print("\nAverage Macro F1:", round(df["Macro F1"].mean(), 4))

if __name__ == "__main__":
    main()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/694 [00:00<?, ?B/s]


>>> PROCESSING TASK: emotion


...

Some weights of ModernBertForSequenceClassification were not initialized from the model checkpoint at answerdotai/ModernBERT-base and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Freezing embeddings + first 13 encoder layers...


Epoch,Training Loss,Validation Loss,Macro F1,Accuracy
1,No log,0.963138,0.332827,0.521739
2,0.920800,0.922094,0.310199,0.521739
3,0.756300,0.915003,0.437769,0.550725


Evaluating stance_hillary on Test Set...




########################################
FINAL RESULTS (Batch Size: 16)
########################################
           Task  Macro F1  Accuracy   Loss
        emotion    0.7147    0.7460 0.7384
      sentiment    0.6784    0.6824 0.7536
           hate    0.5116    0.5364 1.8050
          irony    0.6310    0.6543 0.6812
      offensive    0.7880    0.8395 0.3714
          emoji    0.3056    0.4366 1.9258
stance_abortion    0.5530    0.6750 0.7642
 stance_atheism    0.5935    0.6955 0.6742
 stance_climate    0.4409    0.7396 0.6748
 stance_hillary    0.4826    0.6441 0.8483

Saved results to modernbert_results_bs16.csv

Average Macro F1: 0.5699


#Inference

In [17]:
import torch
import os
import glob
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch.nn.functional as F


# Task labels I trained on
TASK_LABELS = {
    "emotion": {0: "anger", 1: "joy", 2: "optimism", 3: "sadness"},
    "sentiment": {0: "negative", 1: "neutral", 2: "positive"},
    "hate": {0: "not-hate", 1: "hate"},
    "irony": {0: "non-irony", 1: "irony"},
    "offensive": {0: "not-offensive", 1: "offensive"},
    "emoji": {
        0: "❤ (red_heart)", 1: "😍 (heart_eyes)", 2: "😂 (tears_of_joy)", 3: "💕 (two_hearts)",
        4: "🔥 (fire)", 5: "😊 (smiling)", 6: "😎 (sunglasses)", 7: "✨ (sparkles)",
        8: "💙 (blue_heart)", 9: "😘 (blowing_kiss)", 10: "📷 (camera)", 11: "🇺🇸 (US_flag)",
        12: "☀ (sun)", 13: "💜 (purple_heart)", 14: "😉 (wink)", 15: "💯 (100)",
        16: "😁 (beaming)", 17: "🎄 (tree)", 18: "📸 (camera_flash)", 19: "😜 (tongue_wink)"
    },
    # Stance tasks (including feminism) use these 3 labels
    "stance": {0: "none", 1: "against", 2: "favor"}
}

TASKS = [
    'emotion', 'sentiment', 'hate', 'irony', 'offensive', 'emoji',
    'stance_abortion', 'stance_atheism', 'stance_climate',
    'stance_feminism', 'stance_hillary'
]

BASE_PATH = "./results"  # Where the previous script saved the models

def get_model_path(task_name):
    """Finds the saved checkpoint for a specific task."""
    task_dir = os.path.join(BASE_PATH, task_name)
    # Trainer saves in subfolders like 'checkpoint-123'.
    checkpoints = glob.glob(f"{task_dir}/checkpoint-*")
    if not checkpoints:
        return None
    # Sort to get the latest checkpoint (highest number)
    checkpoints.sort(key=lambda x: int(x.split('-')[-1]))
    return checkpoints[-1]


In [18]:
def analyze_tweet(tweet_text):
    print(f"\n{'='*60}")
    print(f"ANALYZING TWEET: \"{tweet_text}\"")
    print(f"{'='*60}")

    tokenizer = AutoTokenizer.from_pretrained("answerdotai/ModernBERT-base")
    inputs = tokenizer(tweet_text, return_tensors="pt", truncation=True, max_length=128)

    device = "cuda" if torch.cuda.is_available() else "cpu"
    inputs = {k: v.to(device) for k, v in inputs.items()}

    results = []

    for task in TASKS:
        model_path = get_model_path(task)

        if not model_path:
            # Only print warning if you really need to debug
            # print(f"⚠️  No model found for task '{task}'")
            continue

        try:
            model = AutoModelForSequenceClassification.from_pretrained(model_path)
            model.to(device)
            model.eval()
        except Exception:
            continue

        with torch.no_grad():
            outputs = model(**inputs)
            logits = outputs.logits
            probs = F.softmax(logits, dim=-1)

            pred_idx = torch.argmax(probs, dim=-1).item()
            confidence = probs[0][pred_idx].item()

            # Select Mapping
            if "stance" in task:
                mapping = TASK_LABELS["stance"]
            else:
                mapping = TASK_LABELS.get(task, {})

            # Get the word (e.g., "Joy")
            label_str = mapping.get(pred_idx, f"LABEL_{pred_idx}")

            print(f"[{task.upper()}]: {label_str} (Confidence: {confidence:.1%})")
            results.append((task, label_str, confidence))

        # Cleanup
        del model
        torch.cuda.empty_cache()

    return results

In [19]:

# ENTER TEST TWEET BELOW

my_test_tweet = "I absolutely love this new update! It makes everything so much easier."

# Run analysis
analyze_tweet(my_test_tweet)


ANALYZING TWEET: "I absolutely love this new update! It makes everything so much easier."
[EMOTION]: joy (Confidence: 95.4%)
[SENTIMENT]: positive (Confidence: 99.9%)
[HATE]: not-hate (Confidence: 99.9%)
[IRONY]: irony (Confidence: 98.3%)
[OFFENSIVE]: not-offensive (Confidence: 99.8%)
[EMOJI]: 😍 (heart_eyes) (Confidence: 45.2%)
[STANCE_ABORTION]: none (Confidence: 45.7%)
[STANCE_ATHEISM]: none (Confidence: 51.6%)
[STANCE_CLIMATE]: none (Confidence: 59.2%)
[STANCE_HILLARY]: none (Confidence: 57.5%)


[('emotion', 'joy', 0.9537299275398254),
 ('sentiment', 'positive', 0.9991195797920227),
 ('hate', 'not-hate', 0.9992314577102661),
 ('irony', 'irony', 0.9826723337173462),
 ('offensive', 'not-offensive', 0.9980823993682861),
 ('emoji', '😍 (heart_eyes)', 0.45236536860466003),
 ('stance_abortion', 'none', 0.4566825032234192),
 ('stance_atheism', 'none', 0.5162567496299744),
 ('stance_climate', 'none', 0.5917286276817322),
 ('stance_hillary', 'none', 0.5749064087867737)]